In [0]:
%sql
-- Join table for CUSTOMER INFO and convert column to user-friendly name 
CREATE OR REPLACE VIEW gold.dim_customer AS
SELECT ROW_NUMBER() OVER (ORDER BY cst_id) AS customer_key,
       ci.cst_id AS customer_id,
       ci.cst_key AS customer_number,
       ci.cst_firstname AS first_name,
       ci.cst_lastname AS last_name,
       la.cntry AS country,
       ci.cst_marital_status AS marital_status,
 CASE WHEN cst_gndr != 'n/a' THEN cst_gndr    -- CRM is the Master table 
      ELSE COALESCE(gen, 'n/a')
END AS gender,
       ca.bdate AS brithdate,
       ci.cst_create_date AS create_date
FROM silver.crm_cust_info ci 
LEFT JOIN silver.erp_cust_az12 ca 
ON ci.cst_key = ca.CID 
LEFT JOIN silver.erp_loc_a101 la  
ON ci.cst_key = la.CID; 


In [0]:
%sql
-- contains historial info and current info. In this case we dont include all the historial info, only stay with current data where end_dt = null 
CREATE OR REPLACE VIEW gold.dim_product AS

SELECT  
row_number() OVER (ORDER BY pn.prd_start_dt, pn.prd_key) AS product_key, -- generate PK for each product
pn.prd_key AS product_number, 
pn.prd_id AS product_id,
pn.prd_nm AS product_name, 
pn.cat_id AS category_id,
pc.cat AS category,
pc.subcat AS subcategory,
pc.maintenance AS maintenance,
pn.prd_cost AS product_cost,
pn.prd_line AS product_line,
pn.prd_start_dt AS product_start_date
FROM silver.crm_prd_info pn 
LEFT JOIN silver.erp_px_cat_g1v2 pc  -- collect all the information about product from 2 sources system 
ON pn.cat_id = pc.ID 
WHERE prd_end_dt IS NULL; -- Filter out all historical data 

-- dimension table 
 

In [0]:
%sql
CREATE OR REPLACE VIEW gold.fact_sales AS 
SELECT sd.sls_ord_num AS order_number,
       pr.product_key,  -- geting surrogate key 
       cu.customer_key, -- getting surrogate key into fact to connect fact with dimension
       
       sd.sls_order_dt AS order_date, 
       sd.sls_ship_dt AS ship_date,
       sd.sls_due_dt AS due_date,
       sd.sls_sales AS sales_amount,
       sd.sls_quantity AS quantity,
       sd.sls_price AS price
      
FROM silver.crm_sales_details sd
LEFT JOIN gold.dim_product pr
ON sd.sls_prd_key = pr.product_number 
LEFT JOIN gold.dim_customer cu
ON sd.sls_cust_id = cu.customer_id